In [ ]:
import os


def get_hf_token():
    """Récupère le HF_TOKEN depuis Colab userdata ou os.environ."""

    # Priorité 1 : os.environ (CLI Colab, local, CI/CD)
    token = os.environ.get("HF_TOKEN")

    if token:
        print("✅ HF_TOKEN trouvé dans os.environ")
        return token

    # Priorité 2 : Colab userdata (interface web)
    try:
        from google.colab import userdata

        token = userdata.get("HF_TOKEN")

        if token:
            print("✅ HF_TOKEN trouvé dans userdata")
            return token

    except (ImportError, Exception):
        pass

    # Aucun token trouvé
    raise ValueError(
        "❌ HF_TOKEN introuvable.\n"
        "Solutions :\n"
        "  - CLI Colab  : colab exec --env HF_TOKEN ...\n"
        "  - Web Colab  : ajoutez le secret via l'icône 🔑\n"
        "  - Local      : export HF_TOKEN=hf_..."
    )


HF_TOKEN = get_hf_token()

# Propagation (utile si vous lancez des sous-processus)
os.environ["HF_TOKEN"] = HF_TOKEN

print(f"   Valeur : {HF_TOKEN[:6]}...{HF_TOKEN[-4:]}")

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Synobotix/llm-dataset.git"
REPO_NAME = "llm-dataset"

PROJECT_DIR = Path(f"/content/{REPO_NAME}")

if PROJECT_DIR.exists():
    print(f"📁 Repo déjà cloné — pull")
    subprocess.run(["git", "pull"], cwd=PROJECT_DIR, check=True)
else:
    print("📥 Clonage...")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)

print(f"✅ Projet : {os.getcwd()}")

In [ ]:
import os
import subprocess

subprocess.run(
    "curl -sSL https://install.python-poetry.org | python3 -",
    shell=True,
    check=True,
)

os.environ["PATH"] = (
    f"{os.path.expanduser('~')}/.local/bin:"
    + os.environ["PATH"]
)

subprocess.run(["poetry", "--version"], check=True)

In [ ]:
import subprocess

# 1. Installer les dépendances Poetry
subprocess.run(
    ["poetry", "install", "--no-root"],
    check=True,
)

# 2. Désinstaller torch CPU
subprocess.run(
    ["poetry", "run", "pip", "uninstall", "-y",
     "torch", "torchvision", "torchaudio"],
    check=True,
)

# 3. Installer torch CUDA 12.1
subprocess.run(
    ["poetry", "run", "pip", "install",
     "torch", "torchvision", "torchaudio",
     "--index-url", "https://download.pytorch.org/whl/cu121"],
    check=True,
)

# 4. Vérifier
result = subprocess.run(
    ["poetry", "run", "python", "-c",
     "import torch; print('torch', torch.__version__); "
     "print('cuda', torch.cuda.is_available()); "
     "print('gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')"],
    capture_output=True,
    text=True,
)
print(result.stdout)

In [ ]:
from huggingface_hub import login, whoami

login(token=HF_TOKEN, add_to_git_credential=False)
print(f"✅ HF : {whoami(token=HF_TOKEN)['name']}")

In [ ]:
!python -m scripts.hub.hub_download

In [ ]:
!python -m scripts.nemotron.download_and_prepare

In [ ]:
!python -m scripts.split_dataset

In [ ]:
!python -m src.tokenizer.train_tokenizer

In [ ]:
!python -m scripts.tokenize_dataset

In [ ]:
!python -m scripts.bitnet.train_bitnet

In [ ]:
!python -m scripts.hub.sync